### Part 1: Faulhaber Factorization and Binomial Thue Equations

For even values of $k$ between $4$ and $10$, we assume the exponent $n$ is an odd prime. Faulhaber's formula allows us to write the sum of consecutive powers as $$x(x+1)(2x+1)T_k(x) = C_k p^\alpha y^n.$$ Because the linear factors $x, x+1,$ and $2x+1$ are pairwise coprime, the prime $p$ divides at most one of them. Dividing out the factor divisible by $p$ and $T_k(x)$ reduces the problem to solving a finite set of binomial Thue equations of the form $Aa^n - Bb^n = \pm 1$. The first section of the script calculates the possible values for the coefficients $A$ and $B$ using the divisibility conditions derived in Section 2 of the manuscript.

In [ ]:
import itertools
from itertools import chain, combinations, product

# Declare global SageMath variables
var('u', 'w')
var('n', 'c')

# =============================================================================
# Section 1: Polynomial Generation and Divisibility Conditions
# =============================================================================

def S(k):
    """
    Generates S_k(x) = 1^k + 2^k + ... + x^k as a polynomial in x 
    using Faulhaber's formula.
    
    Args:
        k (int): The exponent.
        
    Returns:
        Polynomial: S_k(x) expanded as a polynomial with rational coefficients.
    """
    return (1 / (k + 1)) * sum(
        binomial(k + 1, m) * ((-1)^m) * bernoulli(m) * x^(k - m + 1) 
        for m in [0..k]
    )

def table1data(k):
    """
    Generates the core divisibility data for a given exponent k.
    Extracts C_k and T_k(x) such that S_k(x) = (1/C_k) * (linear factors) * T_k(x).
    
    Args:
        k (int): The exponent (k >= 2).
        
    Returns:
        list: Contains [k, C_k, T_k, T_k(x-1), T_k(u), gcd_u_Tk, gcd_Ck_Tk]
              where u = x(x+1). Returns None if k = 1.
    """
    Sk = factor(S(k))
    evenfac = x * (x + 1) * (2 * x + 1)
    oddfac = x^2 * (x + 1)^2
    Skcoefs = Sk.coefficients()
    
    Ck = lcm(coef[0].denominator() for coef in Skcoefs)
    usubs = (-1 + sqrt(1 + 4 * u)) / 2
    
    if k == 1:
        return None
    elif k % 2 == 0:
        Tk = factor(simplify(Ck * Sk / evenfac))
    else:
        Tk = factor(simplify(Ck * Sk / oddfac))
        
    Tkxp1 = expand(Tk(x = x - 1))
    Tku = expand(Tk(x = usubs))
    Tkcoefs = Tku.coefficients(sparse=False)
    
    # Determine gcd of u and T_k(x)
    evendivs = set()
    if Integer(Tkcoefs[0]) % 2 == 0:
        for d in divisors(Integer(Tkcoefs[0])):
            if d % 2 == 0:
                evendivs.add(d)
    else:
        evendivs = set(divisors(Integer(Tkcoefs[0])))
        
    gcduTk = evendivs
    
    # Determine gcd of C_k and T_k(x)
    gcdCkTk = set(gcd(Tk(x=i), Ck) for i in range(0, Ck))
    
    return [k, Ck, Tk, Tkxp1, Tku, gcduTk, gcdCkTk]

def table2data(k):
    """
    Generates coefficients mapping T_k(x) and v = 2x+1.
    Finds a(x), b(x), and r satisfying a(x)T_k(x) + b(x)v = r.
    
    Args:
        k (int): Even exponent >= 4.
        
    Returns:
        list: [k, a(x), b(x), r]
    """
    Tk = table1data(k)[2]
    Tkv = expand(Tk(x = (w - 1) / 2))
    Tkvcoefs, Tkvcoefs2 = Tkv.coefficients(), Tkv.coefficients(sparse=False)
    
    aval = lcm(coef[0].denominator() for coef in Tkvcoefs)
    rval = aval * Tkvcoefs2[0]
    signr = sgn(rval)
    bval = expand((aval * Tkv - rval) / w)
    
    return [k, signr * aval, bval, Integer(signr * rval)]

def determine_ABC(k):
    """
    Calculates the possible values for the coefficients (A, B, C)
    arising from the coprime factors of the Diophantine equation.
    
    Args:
        k (int): The exponent.
        
    Returns:
        set: A collection of valid (A, B, C) triples.
    """
    print(f"Determining constants for k = {k}...")
    Ck, gcd_Ck_T = table1data(k)[1], table1data(k)[6]
    gcd_T_u = table1data(k)[5]
    gcd_T_v = set(divisors(table2data(k)[3]))
    
    initial_pairs = prelim_pairs(Ck, gcd_Ck_T)
    initial_triples = split_A_B_terms(initial_pairs, gcd_T_u)
    triples = set()
    
    for i in product(initial_triples, gcd_T_v):
        # Filter out triples with common factors based on radical conditions
        if (gcd(i[0][0].denominator(), i[1]) > 1 or 
            gcd(i[0][1].denominator(), i[1]) > 1 or 
            gcd(i[0][0].numerator(), i[1]) > 1 or 
            gcd(i[0][1].numerator(), i[1]) > 1):
            continue
            
        rad_0 = radical(i[0][0].numerator() * i[0][0].denominator())
        rad_1 = radical(i[0][1].numerator() * i[0][1].denominator())
        rad_2 = radical((i[0][2] / i[1]).numerator() * (i[0][2] / i[1]).denominator())
        
        if gcd(rad_0, rad_1) > 1 or gcd(rad_0, rad_2) > 1 or gcd(rad_1, rad_2) > 1:
            continue
            
        triples.add((i[0][0], i[0][1], i[0][2] / i[1]))
        
    return triples

def determine_AB(k):
    """
    Extracts the pairs (A, B) required to solve the resulting 
    binomial Thue equations Aa^n - Bb^n = +/- 1.
    
    Returns:
        tuple: (pairs, third_term_dict) containing the set of (A,B) pairs 
               and a dictionary mapping these to the third term.
    """
    ABC = determine_ABC(k)
    Ck, gcd_Ck_T = table1data(k)[1], table1data(k)[6]
    pairs = set()
    
    print(f"Determining all possible equations Aa^n - Bb^n = +/-1...")
    
    for AB in ABC:
        if Ck / (AB[0].numerator() * AB[1].numerator() * AB[2].numerator()) not in gcd_Ck_T:
            continue
        pairs.add((max(AB[0], AB[1]), min(AB[0], AB[1])))
        pairs.add((max(2 * AB[0], AB[2]), min(2 * AB[0], AB[2])))
        pairs.add((max(2 * AB[1], AB[2]), min(2 * AB[1], AB[2])))
        
    third_term_dict = {pair: set() for pair in pairs}
    
    for AB in ABC:
        if Ck / (AB[0].numerator() * AB[1].numerator() * AB[2].numerator()) not in gcd_Ck_T:
            continue
        third_term_dict[(max(AB[0], AB[1]), min(AB[0], AB[1]))].add(AB[2])
        third_term_dict[(max(2 * AB[0], AB[2]), min(2 * AB[0], AB[2]))].add(AB[1])
        third_term_dict[(max(2 * AB[1], AB[2]), min(2 * AB[1], AB[2]))].add(AB[0])
        
    print(f"Found {len(pairs)} pairs for (A, B).")
    return pairs, third_term_dict

# --- Auxiliary Functions for determine_AB ---

def even_divisors(z):
    return [d for d in divisors(z) if d % 2 == 0]

def prelim_pairs(Ck, gcd_Ck_T):
    prelim_pairs_set = set()
    for d in gcd_Ck_T:
        for d1 in even_divisors(ZZ(Ck / d)):
            if gcd(d1, ZZ(Ck / (d1 * d))) > 1:
                continue
            prelim_pairs_set.add((d1, ZZ(Ck / (d1 * d))))
    return prelim_pairs_set

def split_A_B_terms(pairs, gcd_u_T):
    triples = set()
    temp_pairs_u_T = set()
    for j in gcd_u_T:
        for pair in split_coprime(j):
            temp_pairs_u_T.add(pair)
            
    for ABC in pairs:
        temp_pairs_AB = split_coprime(ABC[0])
        for i in product(temp_pairs_u_T, temp_pairs_AB):
            triples.add((i[1][0] / i[0][0], i[1][1] / i[0][1], ABC[1]))
    return triples

def split_coprime(N):
    """Returns all possible pairs of coprime positive integers (L,M) where L*M = N."""
    pairs = set()
    for i in divisors(N):
        if gcd(i, N / i) == 1:
            pairs.add((i, N / i))
    return pairs

### Part 2: Absolute Upper Bounds via Laurent's Theorem

To restrict the search space for the exponent $n$, we apply Baker's method. For equations where $|A-B| > 1$, we construct a linear form in two logarithms. The script implements a theorem due to Laurent (Lemma 4.2) to bound these forms. This calculation yields an absolute upper bound $n_0$ for the exponent $n$ in each equation $Aa^n - Bb^n = \pm 1$. Cases with $|A-B| = 1$ have at most one positive integer solution (by the work of Bennett and others) and are treated as a distinct case.

In [9]:
# =============================================================================
# Section 2: Linear Forms in Two Logarithms (Applying Laurent's Result)
# =============================================================================

RR = RealField(200)

def sigma(mu):
    return ((1 + 2 * mu - mu^2) / 2).n()

def lambdaval(rho, mu):
    return sigma(mu) * log(rho)

def h(A, B, rho, mu, a1, a2, Deg, hid):
    if hid == 0:
        return Deg * (log(n / a2 + 1 / a1) + log(lambdaval(rho, mu)) + 1.75) + 0.06
    elif hid == 1:
        return max(lambdaval(rho, mu), (Deg * log(2) / 2))

def H(A, B, rho, mu, a1, a2, Deg, hid):
    return h(A, B, rho, mu, a1, a2, Deg, hid) / lambdaval(rho, mu) + 1 / sigma(mu)

def omega(A, B, rho, mu, a1, a2, Deg, hid):
    return 2 * (1 + sqrt(1 + 1 / (4 * H(A, B, rho, mu, a1, a2, Deg, hid)^2)))

def theta(A, B, rho, mu, a1, a2, Deg, hid):
    return omega(A, B, rho, mu, a1, a2, Deg, hid) / 2 - 1 + 1 / (2 * H(A, B, rho, mu, a1, a2, Deg, hid))

def best_rational_near_0(irr2):
    cf_rs = continued_fraction(irr2)
    m, best_rat = 1, []
    while cf_rs.convergent(m) < 1 or cf_rs.convergent(m) > irr2:
        best_rat.append(cf_rs[m])
        m += 1
    return cf_rs.convergent(m)

def bounds(A, B, nlb):
    r1s1, r2s2 = min(A, B), max(A, B)
    trivial_lb = max(radical(r1s1.denominator()), 2)
    try:
        u = exp(log(r2s2 / r1s1) / nlb + 1 / (nlb * r1s1 * (trivial_lb)^nlb))
    except ZeroDivisionError:
        print(f"ZeroDivisionError for the pair A = {A}, B = {B}.")
        u = None
    return u

def c_lb_trivial(A, B):
    r1s1 = min(A, B)
    ub = bounds(A, B, 20000)
    ab = best_rational_near_0(ub)
    maxab_lb = max(ab.numerator(), ab.denominator())
    return max(radical(r1s1.denominator()), maxab_lb)

def Verify_lambda(rho, mu, a1, a2, c_lb):
    return ((a1 * a2)(c = c_lb)).n() >= (lambdaval(rho, mu)^2).n()

def avals(rho, mu, A, B, nlb):
    r1, r2 = min(A, B).numerator(), max(A, B).numerator()
    s1, s2 = min(A, B).denominator(), max(A, B).denominator()
    
    a1 = (rho - 1) * ((1 / nlb) * log((r2 * s1) / (r1 * s2)) + 
         (2 * s1) / (nlb * r1 * max(radical(s1), 2)^nlb)) + 2 * log(c)
    a2 = (rho - 1) * log((r2 * s1) / (r1 * s2)) + 2 * log(r2 * s1)
    
    return a1, a2

def C(rho, mu, A, B, Deg, hid):
    a1, a2 = avals(rho, mu, A, B, Deg)
    pt1 = mu / (lambdaval(rho, mu)^3 * sigma(mu))
    pt2 = omega(A, B, rho, mu, a1, a2, Deg, hid) / 6
    pt3 = (omega(A, B, rho, mu, a1, a2, Deg, hid)^2 / 9 + 
           4/3 * (1/a1 + 1/a2) * lambdaval(rho, mu) * omega(A, B, rho, mu, a1, a2, Deg, hid) / 
           H(A, B, rho, mu, a1, a2, Deg, hid))
    
    pt4num = 8 * lambdaval(rho, mu) * omega(A, B, rho, mu, a1, a2, Deg, hid)^(5/4) * theta(A, B, rho, mu, a1, a2, Deg, hid)^(1/4)
    pt4denom = 3 * sqrt(a1 * a2 * H(A, B, rho, mu, a1, a2, Deg, hid))
    
    return pt1 * (pt2 + 1/2 * sqrt(pt3 + (pt4num / pt4denom)))^2

def Cprime(rho, mu, A, B, Deg, hid):
    Cval = C(rho, mu, A, B, Deg, hid)
    a1, a2 = avals(rho, mu, A, B, Deg)
    
    Cprimenum = Cval * sigma(mu) * omega(A, B, rho, mu, a1, a2, Deg, hid) * theta(A, B, rho, mu, a1, a2, Deg, hid)
    Cprimedenom = lambdaval(rho, mu)^3 * mu
    return sqrt(Cprimenum / Cprimedenom)

def LambdaLB(rho, mu, A, B, Deg, hid):
    a1, a2 = avals(rho, mu, A, B, Deg)
    H_val = h(A, B, rho, mu, a1, a2, Deg, hid) + lambdaval(rho, mu) / sigma(mu)
    
    pt1 = C(rho, mu, A, B, Deg, hid) * (H_val)^2 * a1 * a2
    pt2 = sqrt(omega(A, B, rho, mu, a1, a2, Deg, hid) * theta(A, B, rho, mu, a1, a2, Deg, hid)) * H_val
    pt3 = log(Cprime(rho, mu, A, B, Deg, hid) * (H_val)^2 * a1 * a2)
    
    return -pt1 - pt2 - pt3

def LambdaUB(A, B):
    r1, s1 = min(A, B).numerator(), min(A, B).denominator()
    return log(2 * s1) - n * log(c) - log(r1)

def Laurent(A, B, cbd, rholb, rhoub, mu, Deg):
    rholb10, rhoub10 = int(10 * rholb), int(10 * rhoub)
    minnval, nbd = 100000, {}
    
    for rho_int in range(rholb10, rhoub10 + 1):
        rho = rho_int / 10.0  # Safe decimal math
        
        a1temp, a2temp = avals(rho, mu, A, B, Deg)
        if not Verify_lambda(rho, mu, a1temp, a2temp, cbd):
            continue
            
        for hid in range(2):
            LB, UB = LambdaLB(rho, mu, A, B, Deg, hid), LambdaUB(A, B)
            ineq = (UB - LB)(c = cbd)
            nbd[hid] = floor(ineq.find_root(50, exp(50 * ln(10)))).next_prime()
            
        current_max = max(nbd.values())
        if current_max < minnval:
            minnval = current_max
            
        # Fast early exit
        if minnval < 20000:
            break
            
    if minnval == 100000:
        raise ValueError(f"Unable to obtain an upper bound on n in the case A = {A}, B = {B}.")
        
    return max(minnval, 20000)

### Part 3: Local Obstructions and Unconditional Resolution

With an upper bound $n_0$ established, we rule out solutions for primes $n < n_0$ using local methods.  The script searches for auxiliary primes $q = mn + 1$ and checks for solutions to $Aa^n - Bb^n \equiv 1 \pmod q$ within the finite field $\mathbb{F}_q$.  If no solutions exist modulo $q$, the exponent $n$ is eliminated.  For the (small) prime exponents that survive this filter, the script utilizes PARI/GP's `gp.thue` algorithm to compute unconditional solutions.  The `flag=1` argument is passed to ensure the results do not rely on the Generalized Riemann Hypothesis.

In [ ]:
# =============================================================================
# Section 3: Local Methods and Obstructions
# =============================================================================
# Implements the search for auxiliary primes q = mn + 1 to test for solutions 
# to Aa^n - Bb^n = +/- 1 modulo q, as outlined in Section 4.3 of the article.

def return_nth_powers_modulo_q(nval, l):
    """Returns a set containing the n-th powers modulo q = 2*l*n + 1."""
    q = 2 * l * nval + 1
    nthpowers = {0}
    gen_mod_q = GF(q).primitive_element()
    
    for i in range(l):
        nthpowers.add(ZZ(gen_mod_q^(nval * i)))
    return nthpowers

def get_mu_m(q, nnval):
    """
    Returns the set mu_m(Fq), which represents all m-th roots of unity 
    in Fq, where m = (q - 1) / n.
    """
    m = (q - 1) / nnval
    mu_m = GF(q)(1).nth_root(ZZ(m), all=True)
    mu_m.append(0)
    return mu_m

def next_prime_1_mod(lb, nval):
    """Returns the smallest prime greater than lb and congruent to 1 mod n."""
    l = ceil(lb / (2 * nval))
    while True:
        q = 2 * l * nval + 1
        if q.is_prime():
            return q
        l += 1

def find_xp(A, B, mu_m, q, nval):
    """
    Generates the set X'_q(A,B) consisting of pairs of m-th roots 
    of unity in Fq that satisfy A(delta_1) - B(delta_2) = 1.
    """
    Xset = set()
    A_val, B_val = A.numerator() * A.denominator(), B.numerator() * B.denominator()
    
    if A_val % q == 0 or B_val % q == 0:
        if A_val % q == 0 and GF(q)(B) in mu_m:
            for j in mu_m:
                Xset.add((j, -1 / B, 1))
            return Xset
        elif B_val % q == 0 and GF(q)(A) in mu_m:
            for j in mu_m:
                Xset.add((1 / A, j, 1))
            return Xset
        return Xset
        
    Aq, Bq = GF(q)(A), GF(q)(B)
    for delta in mu_m:
        if (Aq * delta - 1) / Bq in mu_m:
            Xset.add((delta, (Aq * delta - 1) / Bq, 1))
    return Xset

def third_term_x(A, B, C, Xp, q, nval, uTdivs, vTgcd, mu_m):
    """
    Evaluates the upsilon^(j) and eta^(j) functions to filter Xp 
    and return X_q^(J_{A,B,C}).
    """
    Xpqtemp = set()
    J = j_sets(A, B, C, uTdivs, vTgcd)
    
    for deltas in Xp:
        for j in J:
            upsilonval = GF(q)(upsilon(GF(q)(A), GF(q)(C), deltas[2], deltas[0], j))
            etaval = GF(q)(eta(GF(q)(B), GF(q)(C), deltas[2], deltas[1], j))
            if upsilonval in mu_m and upsilonval == etaval:
                Xpqtemp.add((deltas[0], deltas[1], upsilonval, deltas[2], j))
    return Xpqtemp

def j_sets(A, B, C, uTdivs, vTgcd):
    """Determines the restricted subset J_{A,B,C} based on divisibility."""
    Aval = A.numerator() * A.denominator()
    Bval = B.numerator() * B.denominator()
    Cval = C.numerator() * C.denominator()
    
    maxuTgcd = max(uTdivs)
    only_v_divs = vTgcd / gcd(vTgcd, maxuTgcd)
    
    if gcd(Aval, only_v_divs) > 1 or B.numerator() % 4 == 0 or (C.numerator() % 2 == 0 and B.numerator() % 2 == 0):
        return {1, 2, 3} - {2, 3}
    elif gcd(Bval, only_v_divs) > 1 or A.numerator() % 4 == 0 or (C.numerator() % 2 == 0 and A.numerator() % 2 == 0):
        return {1, 2, 3} - {1, 3}
    elif gcd(Cval, only_v_divs) > 1:
        return {1, 2, 3} - {1, 2}
    return {1, 2, 3}

# --- Auxiliary Functions (upsilon, eta, omega) ---

def upsilon(A, C, pm1, z, j):
    if j == 1: return QQ(QQ(A) * QQ(z) + 1) / QQ(2 * C)
    elif j == 2: return QQ(QQ(A) * QQ(z) / 2 - 1) / QQ(C)
    elif j == 3: return QQ(2 * QQ(A) * QQ(z) - 1) / QQ(C)
    raise ValueError(f"Invalid j={j}. Must be 1, 2, or 3.")

def eta(B, C, pm1, z, j):
    if j == 1: return (B * z / 2 + 1) / C
    elif j == 2: return (B * z - 1) / (2 * C)
    elif j == 3: return (2 * B * z + 1) / C
    raise ValueError(f"Invalid j={j}. Must be 1, 2, or 3.")

def omegafunc(A, D, pm1, z, j, Tk):
    if j == 1: return Tk(x = QQ(QQ(A) * QQ(z))) / QQ(D)
    elif j == 2: return Tk(x = QQ(QQ(A) * QQ(z) / 2 - 1)) / QQ(D)
    elif j == 3: return Tk(x = QQ(QQ(A) * QQ(z) - 1)) / QQ(D)
    elif j == 4: return Tk(x = QQ(QQ(A) * QQ(z) / 2)) / QQ(D)
    raise ValueError(f"Invalid j={j}. Must be 1, 2, 3, or 4.")

### Part 4: Thue Equation Solvers and Modulo Reductions

This section resolves the surviving binomial Thue equations. For equations where the bounds derived from linear forms in logarithms and local obstructions leave unresolved prime exponents, we apply reduction modulo $n^2$ to identify further contradictions. Equations that bypass these methods are solved unconditionally using PARI/GP's `gp.thue` function to ensure no solutions are missed.

In [8]:
# =============================================================================
# Section 4: Thue Equation Solvers and Modulo Reductions
# =============================================================================

R.<X,Y> = PolynomialRing(ZZ)

def solve_thue(f, m):
    """
    Unconditionally solves the Thue equation f without relying on the 
    Generalized Riemann Hypothesis by setting flag=1 in PARI/GP.
    """
    assert f.is_homogeneous()
    parithueinit = gp.thueinit(f.subs({f.variables()[1]: 1}), flag=1)
    return gp.thue(parithueinit, m).sage()

def check_mod_n_squared(Aval, Bval, nval, mval):
    """Attempts to rule out solutions modulo n^2 as a fallback method."""
    nth_pows = nth_powers_mod_n_squared(nval, mval)
    sols = set()
    A = Mod(Aval.numerator() * Aval.denominator()^(nval - 1), nval^mval)
    B = Mod(Bval.numerator() * Bval.denominator()^(nval - 1), nval^mval)
    
    for j in product(nth_pows, repeat=2):
        if Mod(A * j[0] - B * j[1], nval^mval) == Mod(1, nval^mval):
            sols.add((j[0], j[1]))
    return sols

def nth_powers_mod_n_squared(nval, mval):
    """Returns all n-th powers modulo n^m."""
    return {Mod(j^nval, nval^mval) for j in range(nval^(mval - 1))}

def diff1_check_for_n(A, B):
    """Checks for values of n for which r_1*s_1^(n-1) - r_2*s_2^(n-1) = +/- 1."""
    r1, s1 = A.numerator(), A.denominator()
    r2, s2 = B.numerator(), B.denominator()
    nvals = set()
    
    nmin = RR(log(r1 / r2 - 1 / (r2 * s1^2)) / log(s2 / s1)).ceiling()
    nmax = RR(log(r1 / r2 + 1 / (r2 * s1^2)) / log(s2 / s1)).floor()
    
    if nmin <= nmax:
        for nval in range(max(nmin, 1), nmax + 1):
            if ZZ(nval + 1).is_prime() and abs(r1 * s1^(nval) - r2 * s2^nval) == 1:
                nvals.add(ZZ(nval + 1))
    return nvals

def laurent_bound_dict(k, ABs):
    print("Applying Laurent's theorem to bound n for |A-B| != 1...")
    bound_dict = {}
    count = 0
    
    for AB in ABs:
        if abs(AB[0] - AB[1]) == 1:
            bound_dict[AB] = 0
            continue
            
        cbd = c_lb_trivial(AB[0], AB[1])
        count += 1
        
        bound_dict[AB] = Laurent(AB[0], AB[1], cbd, 2, 20, 1/3, 1)
        print(f"Bound for (A,B) = ({AB[0]},{AB[1]}) is n < {bound_dict[AB]}. ({count} pairs processed)", end='\r')
        
    print() # Newline after progress bar
    return bound_dict

### Part 5: Equation Verification

Once integer candidate values for $x$ are identified from the above outputs, this script evaluates the candidates against the original generalized cannonball equation. It computes $S_k(x)$ using Faulhaber's formula and verifies whether the integer factors into the required form $p^\alpha y^n$, isolating the valid solutions.

In [ ]:
# =============================================================================
# Section 5: Equation Verification
# =============================================================================

def evaluate_and_print_solution(k, x_val):
    """
    Helper function to evaluate S_k(x) and print formatted results if 
    the solution matches the Diophantine equation parameters.
    """
    Sval = S(k)(x = x_val)
    p, alpha, y, n_found = solution_check(Sval)
    
    if p != 0:
        n_str = "n" if y == 1 else str(n_found)
        msg = f"Solution with k = {k}, x = {x_val}, y = {y}, n = {n_str}, p = {p}, alpha = {alpha}."
        print(f"\r{msg.ljust(80)}")

def diff1eqns_check(A, B, k):
    """Checks equations where |A - B| = 1 to map to S_k(x) = p^alpha y^n."""
    if A > B:
        if A % 4 == 0:
            evaluate_and_print_solution(k, (A - 2) / 2)
        elif A % 4 == 2:
            evaluate_and_print_solution(k, (A - 2) / 2)
            evaluate_and_print_solution(k, A - 1)
        elif A % 2 == 1:
            evaluate_and_print_solution(k, A - 1)
            
    elif A < B:
        if A % 4 == 0:
            evaluate_and_print_solution(k, A / 2)
        elif A % 4 == 2:
            evaluate_and_print_solution(k, A / 2)
            evaluate_and_print_solution(k, A)
        elif A % 2 == 1:
            evaluate_and_print_solution(k, A)

def general_check(Aval, Bval, a, b, k, nval):
    """Maps computed Thue solutions back to valid parameters for S_k(x)."""
    A = Aval * a^nval
    B = Bval * b^nval
    
    if A > B:
        if A % 2 == 0:
            evaluate_and_print_solution(k, (A - 2) / 2)
            evaluate_and_print_solution(k, A - 1)
        elif A % 2 == 1:
            evaluate_and_print_solution(k, A - 1)
            
    elif A < B:
        if A % 2 == 0:
            evaluate_and_print_solution(k, A / 2)
            evaluate_and_print_solution(k, A)
        elif A % 2 == 1:
            evaluate_and_print_solution(k, A)

def solution_check(sol):
    """
    Analyzes an evaluated S_k(x) to verify if it forms a near-solution 
    of the form p^alpha y^n.
    
    Returns:
        tuple: (p, alpha, y, n) if valid, otherwise (0, 0, 0, 0).
    """
    if sol in (0, 1):
        return 0, 0, 0, 0
        
    sol_zz = ZZ(sol)
    if sol_zz.is_perfect_power():
        return 0, 0, 0, 0
    elif sol_zz.is_prime():
        return sol_zz, 1, 1, 1
        
    facsol = list(factor(sol_zz))
    for i in range(len(facsol)):
        base_term = sol_zz / (facsol[i][0]^facsol[i][1])
        if ZZ(base_term).is_perfect_power():
            exponent_list = [facsol[j][1] for j in range(len(facsol)) if j != i]
            nval = gcd_list(exponent_list)
            return facsol[i][0], facsol[i][1], ZZ(base_term)^(1 / nval), nval
            
    return 0, 0, 0, 0

def gcd_list(int_list):
    """Computes the greatest common divisor for a list of integers."""
    d = int_list[0]
    for term in int_list[1:]:
        d = gcd(term, d)
    return d

def solve_sk(k):
    """Executes the full pipeline to solve S_k(x) = p^\alpha * y^n."""
    print(f"Solving the equation {factor(S(k))} = p^alpha * y^n...")
    return solve_many_thue(k)

### Part 6: Orchestration Pipeline

This block combines the local checks and the Thue equation solvers into a unified execution pipeline. It bounds $n$ using Laurent's result on linear forms in two logarithms, applies the local methods to eliminate primes $n < n_0$, and routes any surviving prime exponents directly to the unconditional Thue solver, completing the computational proof for each exponent $k$.

In [ ]:
# =============================================================================
# Section 6: Orchestration (Local Method Checks and Thue Equation Pipeline)
# =============================================================================

def tk_x(q, A, B, D, mu_m, Xp, uTdivs, vTgcd, Tk):
    """Applies the T_k test to filter potential solutions modulo q."""
    XT, maxuTgcd = set(), max(uTdivs)
    only_v_divs = vTgcd / gcd(vTgcd, maxuTgcd)
    
    for deltas in Xp:
        if A.numerator() % 2 == 0 and deltas[2] == 1:
            if A.numerator() % 4 == 0 or gcd(B.denominator(), only_v_divs) > 1:
                dn = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 2, Tk))
                if dn in mu_m: return {dn}
            else:
                dn1 = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 2, Tk))
                dn2 = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 3, Tk))
                for d in [dn1, dn2]:
                    if d in mu_m: return {d}
                    
        elif A.numerator() % 2 == 0 and deltas[2] == -1:
            if A.numerator() % 4 == 0 or gcd(B.denominator(), only_v_divs) > 1:
                dn = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 4, Tk))
                if dn in mu_m: return {dn}
            else:
                dn1 = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 1, Tk))
                dn2 = GF(q)(omegafunc(GF(q)(A), GF(q)(D), deltas[2], deltas[0], 4, Tk))
                for d in [dn1, dn2]:
                    if d in mu_m: return {d}
                    
        elif A.numerator() % 2 == 1 and deltas[2] == 1:
            if B.numerator() % 4 == 0 or gcd(A.denominator(), only_v_divs) > 1:
                dn = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 4, Tk))
                if dn in mu_m: return {dn}
            else:
                dn1 = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 1, Tk))
                dn2 = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 4, Tk))
                for d in [dn1, dn2]:
                    if d in mu_m: return {d}
                    
        elif A.numerator() % 2 == 1 and deltas[2] == -1:
            if B.numerator() % 4 == 0 or gcd(A.denominator(), only_v_divs) > 1:
                dn = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 2, Tk))
                if dn in mu_m: return {dn}
            else:
                dn1 = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 2, Tk))
                dn2 = GF(q)(omegafunc(GF(q)(B), GF(q)(D), deltas[2], deltas[1], 3, Tk))
                for d in [dn1, dn2]:
                    if d in mu_m: return {d}
    return XT

def find_d(k, A, B, C, C_k, gcdsTkCk):
    """Finds the possible values for the constant D given A, B, and C."""
    Ckparts, Dset = set(), set()
    Dtemp = A.denominator() * B.denominator()
    
    if (A.numerator() * B.numerator()) % 4 == 0:
        Ckpart = 2 * C_k / (A.numerator() * B.numerator())
    else:
        Ckpart = C_k / (A.numerator() * B.numerator())
        
    if Ckpart not in ZZ:
        raise ValueError("The extracted C_k part is not an integer.")
        
    for div in ZZ(Ckpart).divisors():
        if div in gcdsTkCk:
            Ckparts.add(div)
            
    if not Ckparts:
        return 0
        
    for cd in Ckparts:
        Dset.add(cd * Dtemp)
        
    return Dset

def local_methods(k, lb, ub, A, B, third_terms, uTdivs, vTgcd, C_k, gcdsTkCk, Tk):
    """
    Applies the local methods described in Section 3.2 to the binomial 
    Thue equation Aa^n - Bb^n = 1 for primes n in (lb, ub).
    """
    nval = lb
    noprimeset = {3}
    
    while nval < ub:
        print(f"pair:{(A,B)}, current n: {nval}", end='\r')
        q = next_prime_1_mod(nval, nval)
        prime_count = 1
        
        while prime_count < 20:
            if A.numerator() * A.denominator() % q == 0 or B.numerator() * B.denominator() % q == 0:
                q = next_prime_1_mod(q, nval)
                continue
                
            mu_m = get_mu_m(q, nval)
            sol = 0
            Xp = find_xp(A, B, mu_m, q, nval)
            
            if not Xp:
                nval = nval.next_prime()
                break
                
            for C in third_terms:
                if C.numerator() * C.denominator() % q == 0:
                    sol = 1
                    break
                    
                Xpq = third_term_x(A, B, C, Xp, q, nval, uTdivs, vTgcd, mu_m)
                if Xpq:
                    sol = 1
                    break
                    
                Ds = find_d(k, A, B, C, C_k, gcdsTkCk)
                if Ds == 0:
                    continue
                    
                for D in Ds:
                    if D % q == 0:
                        sol = 1
                        break
                    XTk = tk_x(q, A, B, D, mu_m, Xp, uTdivs, vTgcd, Tk)
                    if XTk:
                        sol = 1
                        break
                        
                if sol == 1:
                    break
                    
            if sol == 0:
                nval = nval.next_prime()
                break
            else:
                prime_count += 1
                q = next_prime_1_mod(q, nval)
                
        if prime_count == 20:
            noprimeset.add(nval)
            nval = nval.next_prime()
            
    return noprimeset

def local_method_check(k):
    """
    For a given even exponent k, computes all possible (A, B) pairs 
    and applies local methods to solve Aa^n - Bb^n = 1 for odd primes n.
    """
    ABs_with_third_term = determine_AB(k)
    ABs, third_term_dict = ABs_with_third_term[0], ABs_with_third_term[1]
    
    table1 = table1data(k)
    uTdivs, C_k, gcdsTkCk, Tk = table1[5], table1[1], table1[6], table1[2]
    vTgcd = table2data(k)[3]
    
    nBounds = laurent_bound_dict(k, ABs)
    badn = {}
    
    print("\nApplying local methods to solve as many equations as possible...")
    count = 1
    
    for AB in ABs:
        if abs(AB[0] - AB[1]) == 1:
            diff1eqns_check(AB[0], AB[1], k)
            continue
            
        elif (AB[0].denominator() > 1 or AB[1].denominator() > 1) and AB[0] != 1 and AB[1] != 1:
            diff1_n = diff1_check_for_n(AB[0], AB[1])
            for nval in diff1_n:
                A_scaled = AB[0].numerator() * AB[0].denominator()^(nval - 1)
                B_scaled = AB[1].numerator() * AB[1].denominator()^(nval - 1)
                diff1eqns_check(A_scaled, B_scaled, k)
                
        elif AB[0] == 1 or AB[1] == 1:
            phiB = radical(euler_phi(radical(AB[0].numerator() * AB[0].denominator() * AB[1].numerator() * AB[1].denominator()))).odd_part()
            tempset, tempset1 = set(), set()
            
            for f in factor(phiB):
                tempset1.add(f[0])
                
            for nval in tempset1:
                third_terms = third_term_dict[AB]
                tempset2 = local_methods(k, nval, nval, AB[0], AB[1], third_terms, uTdivs, vTgcd, C_k, gcdsTkCk, Tk)
                if tempset2:
                    tempset.add(nval)
                    
            badn[AB] = tempset
            continue
            
        third_terms = third_term_dict[AB]
        badn[AB] = local_methods(k, 5, nBounds[AB], AB[0], AB[1], third_terms, uTdivs, vTgcd, C_k, gcdsTkCk, Tk)
        print(f"Progress: Applied local methods to {count} binomial Thue equations so far.", end='\r')
        count += 1
        
    return badn

def solve_many_thue(k):
    """
    Attempts to solve remaining equations by reducing mod n^2, 
    or unconditionally via gp.thue if n <= 5. Returns a dictionary 
    of equations unable to be solved this way.
    """
    eqns = local_method_check(k)
    tempdict = {}
    
    print("\nUsing the PARI/GP Thue equation solver to resolve remaining equations...")
    
    for AB in eqns.keys():
        bignset = set()
        m = AB[0].denominator() * AB[1].denominator()
        A, B = AB[0] * m, AB[1] * m
        
        if A == 0 and B == 0:
            continue
            
        for nval in eqns[AB]:
            if nval > 5:
                bignset.add(nval)
                continue
                
            if not check_mod_n_squared(AB[0], AB[1], nval, 2):
                continue
                
            print(f"Attempting to solve Thue equation: {AB[0]}*X^{nval} - {AB[1]}*Y^{nval} = 1".ljust(80), end='\r')
            sols = solve_thue(A * X^nval - B * Y^nval, m)
            
            for sol in sols:
                if sol[0] == 0 or sol[1] == 0:
                    continue
                elif sol[0] % AB[0].denominator() == 0 and sol[1] % AB[1].denominator() == 0:
                    general_check(AB[0], AB[1], sol[0], sol[1], k, nval)
                    
        if bignset:
            tempdict[AB] = bignset
            
    if tempdict:
        print(f"\nUnable to solve the following pairs (A, B) and exponents n:\n{tempdict}")
        
    return tempdict

In [10]:
%time solve_sk ( 2 )

Solving the equation 1/6*(2*x + 1)*(x + 1)*x = p^alpha * y^n...
Determining constants for k = 2...
Determining all possible equations Aa^n - Bb^n = +/-1...
Found 6 pairs for (A, B).
Applying Laurent's theorem to bound n for |A-B| != 1...
Bound for (A,B) = (4,1) is n < 20000. (3 pairs processed))

Applying local methods to solve as many equations as possible...
Solution with k = 2, x = 2, y = 1, n = n, p = 5, alpha = 1.                     

Using the PARI/GP Thue equation solver to resolve remaining equations...
CPU times: user 147 ms, sys: 12.4 ms, total: 160 ms
Wall time: 160 ms


{}

In [11]:
%time solve_sk ( 4 )

Solving the equation 1/30*(3*x^2 + 3*x - 1)*(2*x + 1)*(x + 1)*x = p^alpha * y^n...
Determining constants for k = 4...
Determining all possible equations Aa^n - Bb^n = +/-1...
Found 36 pairs for (A, B).
Applying Laurent's theorem to bound n for |A-B| != 1...
Bound for (A,B) = (10,1/7) is n < 20000. (31 pairs processed)

Applying local methods to solve as many equations as possible...
Solution with k = 4, x = 2, y = 1, n = n, p = 17, alpha = 1.                    
Solution with k = 4, x = 2, y = 1, n = n, p = 17, alpha = 1.                    
Solution with k = 4, x = 3, y = 7, n = 2, p = 2, alpha = 1.                     
Progress: Applied local methods to 24 binomial Thue equations so far.
Using the PARI/GP Thue equation solver to resolve remaining equations...
Attempting to solve Thue equation: 10*X^3 - 1/7*Y^3 = 1                         
Unable to solve the following pairs (A, B) and exponents n:
{(10, 3): {7}}
CPU times: user 20 s, sys: 448 ms, total: 20.5 s
Wall time: 20.9 s


{(10, 3): {7}}

In [12]:
%time solve_sk ( 6 )

Solving the equation 1/42*(3*x^4 + 6*x^3 - 3*x + 1)*(2*x + 1)*(x + 1)*x = p^alpha * y^n...
Determining constants for k = 6...
Determining all possible equations Aa^n - Bb^n = +/-1...
Found 36 pairs for (A, B).
Applying Laurent's theorem to bound n for |A-B| != 1...
Bound for (A,B) = (21,4) is n < 20000. (32 pairs processed)ed)

Applying local methods to solve as many equations as possible...
Progress: Applied local methods to 25 binomial Thue equations so far.
Using the PARI/GP Thue equation solver to resolve remaining equations...
Attempting to solve Thue equation: 21*X^3 - 4*Y^3 = 1                           
Unable to solve the following pairs (A, B) and exponents n:
{(14, 3): {7}}
CPU times: user 20.8 s, sys: 430 ms, total: 21.2 s
Wall time: 21.5 s


{(14, 3): {7}}

In [13]:
%time solve_sk ( 8 )

Solving the equation 1/90*(5*x^6 + 15*x^5 + 5*x^4 - 15*x^3 - x^2 + 9*x - 3)*(2*x + 1)*(x + 1)*x = p^alpha * y^n...
Determining constants for k = 8...
Determining all possible equations Aa^n - Bb^n = +/-1...
Found 36 pairs for (A, B).
Applying Laurent's theorem to bound n for |A-B| != 1...
Bound for (A,B) = (30,1/127) is n < 20000. (31 pairs processed)

Applying local methods to solve as many equations as possible...
Solution with k = 8, x = 2, y = 1, n = n, p = 257, alpha = 1.                   
Solution with k = 8, x = 2, y = 1, n = n, p = 257, alpha = 1.                   
Progress: Applied local methods to 24 binomial Thue equations so far.
Using the PARI/GP Thue equation solver to resolve remaining equations...
Attempting to solve Thue equation: 30*X^3 - 1/127*Y^3 = 1                       
Unable to solve the following pairs (A, B) and exponents n:
{(10, 3): {7}}
CPU times: user 21.6 s, sys: 487 ms, total: 22.1 s
Wall time: 3min 9s


{(10, 3): {7}}

In [14]:
%time solve_sk ( 10 )

Solving the equation 1/66*(3*x^6 + 9*x^5 + 2*x^4 - 11*x^3 + 3*x^2 + 10*x - 5)*(x^2 + x - 1)*(2*x + 1)*(x + 1)*x = p^alpha * y^n...
Determining constants for k = 10...
Determining all possible equations Aa^n - Bb^n = +/-1...
Found 216 pairs for (A, B).
Applying Laurent's theorem to bound n for |A-B| != 1...
Bound for (A,B) = (2,3/5) is n < 20000. (212 pairs processed)sed))
Applying local methods to solve as many equations as possible...
Solution with k = 10, x = 2, y = 5, n = 2, p = 41, alpha = 1.                   
Solution with k = 10, x = 2, y = 5, n = 2, p = 41, alpha = 1.                   
Solution with k = 10, x = 2, y = 5, n = 2, p = 41, alpha = 1.                   
Progress: Applied local methods to 197 binomial Thue equations so far.
Using the PARI/GP Thue equation solver to resolve remaining equations...
Attempting to solve Thue equation: 2*X^3 - 3/5*Y^3 = 1                          
Unable to solve the following pairs (A, B) and exponents n:
{(2, 3/2555): {7}, (33, 2/5): {7

{(2, 3/2555): {7},
 (33, 2/5): {7},
 (4, 3/2555): {7},
 (4, 1/511): {7},
 (2/5, 1/73): {7},
 (4, 3/365): {7},
 (6, 1/5): {7},
 (11, 6): {7},
 (12, 1/7): {7},
 (3, 2/5): {7}}

### Resolving Outstanding Low-Exponent Cases ($n=7$)

During the execution of the local methods, a small number of binomial Thue equations with exponent $n=7$ routinely survive the elimination process. 

While certain $n=7$ cases with extremely large coefficients (e.g., those divisible by $73^6$ or $2555^6$ when $k=10$) are highly resistant to standard algorithms and require manual resolution via Thue equations obtained by factoring over $\mathbb{Q}(\sqrt{5})$, the equations listed below are computationally tractable. 

Here, we unconditionally resolve these remaining equations using PARI/GP's `gp.thue` algorithm. By utilizing the function defined earlier (which passes `flag=1` to the PARI backend), we ensure that the computation of the fundamental units of the underlying number fields does not rely on the Generalized Riemann Hypothesis (GRH).

In [ ]:
# =============================================================================
# Tractable Thue Equations (n=7)
# =============================================================================

# Define the list of tractable equations that survived the local methods.
# Note: Variables X and Y belong to the PolynomialRing(ZZ) defined earlier.
tractable_n7_equations = [
    12*X^7 - 7^6*Y^7,
    3*X^7 - 2*5^6*Y^7,
    11*X^7 - 6*Y^7,
    6*X^7 - 5^6*Y^7,
    33*X^7 - 2*5^6*Y^7,
    14*X^7 - 3*Y^7,
    10*X^7 - 3*Y^7
]

def resolve_tractable_n7_cases(equations):
    """
    Iterates through the surviving n=7 binomial Thue equations and 
    unconditionally solves them using PARI/GP.
    """
    print("Unconditionally solving the remaining tractable n=7 Thue equations...\n")
    
    for eq in equations:
        eq_str = f"{eq} = 1"
        print(f"Solving: {eq_str.ljust(35)}", end="")
        
        sols = solve_thue(eq, 1)
        
        if not sols:
            print("Solutions: None")
        else:
            print(f"Solutions (X, Y): {sols}")

# Execute the solver
resolve_tractable_n7_cases(tractable_n7_equations)

Unconditionally solving the remaining tractable n=7 Thue equations...

Solving: 12*X^7 - 117649*Y^7 = 1            Solutions: None
Solving: 3*X^7 - 31250*Y^7 = 1              Solutions: None
Solving: 11*X^7 - 6*Y^7 = 1                 Solutions: None
Solving: 6*X^7 - 15625*Y^7 = 1              Solutions: None
Solving: 33*X^7 - 31250*Y^7 = 1             Solutions: None
Solving: 14*X^7 - 3*Y^7 = 1                 Solutions: None
Solving: 10*X^7 - 3*Y^7 = 1                 Solutions: None
